# Decomposing Cliffords into transvections (π/4 Pauli exponents)

Every Clifford symplectic action is an ordered product of **Clifford transvections**, the
`π/4` Pauli exponents $\exp\!\big(i\tfrac{\pi}{4} P_v\big)$. Conjugation by such an exponent acts on
Pauli operators as a **symplectic transvection**

$$
x \;\mapsto\; x + \langle x, v\rangle\, v,
$$

where $\langle\cdot,\cdot\rangle$ is the symplectic (commutation) form. `paulimer` exposes two
decompositions, following the transvection framework of
[arXiv:2102.11380](https://arxiv.org/abs/2102.11380) (Pllaha, Volanto & Tirkkonen,
*Decomposition of Clifford Gates*):

- [`CliffordUnitary.to_transvections`](../paulimer.pyi) — a greedy reduction that always returns a
  **linear** number of factors ($O(n)$),
- [`CliffordUnitary.to_transvections_minimal`](../paulimer.pyi) — the **strict minimum** number of
  unsigned factors, plus a final Pauli correction.

The greedy list reproduces only the unsigned symplectic action.
The minimal API returns `(transvections, pauli_correction)` and reproduces all tableau signs after the final Pauli.
The minimum counts only the rotations, not the final correction. Neither API reconstructs global phase.

In [1]:
import paulimer
from paulimer import CliffordUnitary, SparsePauli, DensePauli

## A single transvection

A `π/4` Pauli exponent *is* a Clifford transvection, so the simplest Cliffords decompose into a
single factor. The phase gate and the Hadamard each need one unsigned transvection.
The returned Pauli correction restores the signs of their conjugation actions:

In [2]:
s_gate = CliffordUnitary.from_name("SqrtZ", [0], qubit_count=1)
hadamard = CliffordUnitary.from_name("Hadamard", [0], qubit_count=1)

print("S        ->", s_gate.to_transvections_minimal())
print("Hadamard ->", hadamard.to_transvections_minimal())

S        -> ([Z], Z)
Hadamard -> ([Y], Z)


## Rebuilding the signed Clifford tableau

Applying the returned transvections in order with
[`left_mul_pauli_exp`](../paulimer.pyi) reconstructs the unsigned symplectic matrix.
Apply `pauli_correction` last with `left_mul_pauli`, not `left_mul_pauli_exp`.
Then compare the full signed tableau. Global phase remains unspecified.

The minimal factor count is either $r$ or $r+1$, where the **residue rank**

$$
r \;=\; 2n - \dim \operatorname{Fix}(F)
$$

is the codimension of the Pauli space fixed up to sign under conjugation.
The dimension $\dim\operatorname{Fix}(F)$ equals `len(clifford.fixed_space())`.

This fixed space is the projective centralizer in the Pauli group with phase quotiented out.
In contrast, `centralizer_of` requires exact commutation.
Clifford `X` fixes `Z` up to sign because `X Z X = -Z`.
Thus `Z` belongs to the fixed space but not to the centralizer of `X`.

In [3]:
def residue_rank(clifford):
    return 2 * clifford.qubit_count - len(clifford.fixed_space())


def rebuild(factors, correction, qubit_count):
    rebuilt = CliffordUnitary.identity(qubit_count)
    for pauli in factors:
        rebuilt.left_mul_pauli_exp(pauli)
    rebuilt.left_mul_pauli(correction)
    return rebuilt


factors, correction = s_gate.to_transvections_minimal()
rebuilt = rebuild(factors, correction, s_gate.qubit_count)
assert rebuilt == s_gate
print("residue rank r      =", residue_rank(s_gate))
print("number of factors   =", len(factors))
print("signed tableau ok  :", rebuilt == s_gate)

residue rank r      = 1
number of factors   = 1
signed tableau ok  : True


## Greedy versus minimal, and the $r+1$ case

For many Cliffords the greedy and minimal decompositions agree, but not always. The CNOT gate has
residue rank $r = 2$ yet needs $r + 1 = 3$ transvections: its symplectic action is *hyperbolic*
($\langle v, vF\rangle = 0$ for all $v$), which forces one extra factor.

In [4]:
cnot = CliffordUnitary.from_name("ControlledX", [0, 1], qubit_count=2)

greedy = cnot.to_transvections()
minimal, correction = cnot.to_transvections_minimal()
assert rebuild(minimal, correction, 2) == cnot
print("residue rank r =", residue_rank(cnot))
print("greedy  :", greedy, " (", len(greedy), "factors )")
print("minimal :", minimal, " (", len(minimal), "factors )")
print("final Pauli:", correction)
print("signed tableau ok:", rebuild(minimal, correction, 2) == cnot)

residue rank r = 2
greedy  : [Z, ZX, IX]  ( 3 factors )
minimal : [Z, ZX, IX]  ( 3 factors )
final Pauli: ZX
signed tableau ok: True


## A subtle case: non-hyperbolic maps that still need $r+1$

The 2021 paper claims that *every* non-hyperbolic Clifford decomposes into exactly $r$ transvections.
That is **not correct over $\mathbb{F}_2$**: some non-hyperbolic maps still require $r + 1$. The
smallest example already occurs on two qubits — the symplectic action built below (a product of the
transvections $X_0, X_1, X_0X_1, Z_0$) has residue rank $r = 3$, is non-hyperbolic, yet needs $4$
transvections. `to_transvections_minimal` returns the correct minimum. See
[`docs/transvection-minimality-correction.md`](../../../docs/transvection-minimality-correction.md)
for the analysis and the limits of the residue fix step.
Default integration tests compare every one- and two-qubit action with a shortest-path oracle.
The three-qubit oracle also replays every returned decomposition, but it is ignored by default.
Run it explicitly with the command in the note. The repository includes no theorem-prover artifact.

In [5]:
example = CliffordUnitary.identity(2)
for pauli in ["X0", "X1", "X0 X1", "Z0"]:
    example.left_mul_pauli_exp(SparsePauli(pauli))

minimal, correction = example.to_transvections_minimal()
assert rebuild(minimal, correction, 2) == example
r = residue_rank(example)
print("residue rank r      =", r)
print("minimal factors     =", minimal, "(", len(minimal), "factors )")
print("needs r + 1          :", len(minimal) == r + 1)
print("final Pauli         =", correction)
print("signed tableau ok  :", rebuild(minimal, correction, 2) == example)

residue rank r      = 3
minimal factors     = [IX, XX, Y, X] ( 4 factors )
needs r + 1          : True
final Pauli         = Z
signed tableau ok  : True


## Summary

- Clifford transvections are `π/4` Pauli exponents. `to_transvections` decomposes any Clifford
  into them with $O(n)$ factors, reproducing its symplectic action.
- `to_transvections_minimal` returns a shortest unsigned list and a final Pauli. Exhaustive tests cover one,
  two and three qubits only. For more qubits this repository states no proof that the residue fix
  step always finds a candidate, and the call panics if the search exhausts every candidate. See
  the correction note for the open question.
- The minimal count is $r$ or $r + 1$, where $r = 2n - \dim\operatorname{Fix}(F)$.
- The final Pauli restores all tableau signs. The unsigned minimum excludes this Pauli.
- Global phase is not reconstructed because the input tableau does not record it.

### References

- T. Pllaha, K. Volanto, O. Tirkkonen, *Decomposition of Clifford Gates*, GLOBECOM 2021,
  [arXiv:2102.11380](https://arxiv.org/abs/2102.11380).
- [`docs/transvection-minimality-correction.md`](../../../docs/transvection-minimality-correction.md)
  — a correction to the paper's minimality claim, with a verified counterexample.